Part 1 for Data Cleaning + Part 2 for EDA

### Part 1.1 Load data and Inspect

In [1]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv(
    "COVID-19BehaviorData_CAN2022.csv",
    low_memory=False
)

print(df_raw.shape)
display(df_raw.head(10))

(6430, 512)


,RecordNo,endtime,qweek,i1_health,i2_health,i7a_health,i3_health,i4_health,i5_health_1,i5_health_2,...,vac_man_96,vac_man_99,q_other,had_covid,vac_boost_beyond,future_1,future_2,had_covid_2,long_covid,future_3
0,41589,06/01/2022 18:02,week 44,,0,,,,,,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,41731,03/01/2022 23:48,week 44,,1,,,,,,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,41756,10/01/2022 04:53,week 44,,0,,,,,,...,Yes,No,"Oui bar , restaurent",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,41846,01/01/2022 13:15,week 44,,5,,,,,,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,41847,01/01/2022 19:12,week 44,,150,,,,,,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,41867,01/01/2022 15:21,week 44,,4,,,,,,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,41868,01/01/2022 20:02,week 44,,14,,,,,,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,41920,02/01/2022 12:36,week 44,,0,,,,,,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,41937,01/01/2022 00:11,week 44,,0,,,,,,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,41951,01/01/2022 00:43,week 44,,5,,,,,,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Part 1.2 Missing value determination

In [2]:
df_raw = df_raw.replace(r"^\s*$", np.nan, regex=True)

missing = df_raw.isna().mean() * 100

display(missing.sort_values(ascending=False).head(15))

ct8_6        100.0
ct5_2        100.0
ct7_2        100.0
ct7_1        100.0
ct6_other    100.0
ct6_99       100.0
ct6_96       100.0
ct6_6        100.0
ct6_5        100.0
ct6_4        100.0
ct6_3        100.0
ct6_2        100.0
ct6_1        100.0
ct5_other    100.0
ct5_99       100.0
dtype: float64

### Part 1.3 Select our target (i9_health) and predictors (15 vars)

In [3]:
target = "i9_health"

num_cols = [
    "age",
    "cantril_ladder"
]

cat_cols = [
    "gender",
    "region",
    "household_size",
    "household_children",
    "employment_status",
    "qweek",
    "PHQ4_1",
    "PHQ4_3",
    "vac2_1",
    "vac2_2",
    "vac7",
    "WCRex1",
    "Vent_3"
]

features = num_cols + cat_cols

df_clean = df_raw[[target] + features].copy()

for col in num_cols:
    df_clean[col] = pd.to_numeric(
        df_clean[col],
        errors="coerce"
)

print("Selected predictors:", len(features))
display(df_clean.head())

Selected predictors: 15


,i9_health,age,cantril_ladder,gender,region,household_size,household_children,employment_status,qweek,PHQ4_1,PHQ4_3,vac2_1,vac2_2,vac7,WCRex1,Vent_3
0,Yes,52,9,Female,Quebec / Québec,4,0,Full time employment,week 44,Not at all,Not at all,5 – Strongly disagree,5 – Strongly disagree,Very much,Somewhat well,Always
1,No,51,6,Female,British Columbia / Colombie Britanique,2,0,Full time employment,week 44,Several days,More than half the days,1 - Strongly agree,2,Moderately,Very well,Always
2,No,57,1,Female,Quebec / Québec,1,0,Part time employment,week 44,Nearly every day,Not at all,5 – Strongly disagree,5 – Strongly disagree,Very much,Very well,Not at all
3,Yes,64,6,Female,Quebec / Québec,1,0,Retired,week 44,More than half the days,More than half the days,1 - Strongly agree,1 - Strongly agree,Moderately,Very badly,Always
4,Yes,45,5,Male,Quebec / Québec,5,5 or more,Full time employment,week 44,Not at all,Not at all,1 - Strongly agree,5 – Strongly disagree,Very much,Somewhat well,Always


### Part 1.4 For Data Encoding & Scaling rules future Modeling
- Age, Cantril Ladder, Household, qweek: Numerical
- Gender: Binary
- Region, Employment: One-Hot Encoding
- PHQ4, Vac2, WCRex1: Ordinal Encoding

In [4]:
# Keep only Yes and No responses
model_df = df_clean[
    df_clean["i9_health"].isin(["Yes", "No"])
].copy()

# Replace missing value as NA for now; impute with appropriate values later
model_df = model_df.replace(["Prefer not to say", "Don't know"], np.nan)

model_df.isna().sum()

i9_health               0
age                     0
cantril_ladder          0
gender                  0
region                  0
household_size        175
household_children     64
employment_status       0
qweek                   0
PHQ4_1                136
PHQ4_3                147
vac2_1                  0
vac2_2                  0
vac7                    0
WCRex1                210
Vent_3                  0
dtype: int64

Convert text-coded numerical variables to numbers:

In [5]:
# Create binary target: No = 1, Yes = 0
model_df["target"] = model_df["i9_health"].map({"Yes": 0, "No": 1})
model_df["gender"] = model_df["gender"].map({"Female": 0, "Male": 1})

# Convert text-coded variables to numeric values
model_df["qweek"] = model_df["qweek"].str.removeprefix("week ").astype(int)

# Remove description for level 1 and 5
model_df["vac2_1"] = model_df["vac2_1"].str[:1].astype(int)
model_df["vac2_2"] = model_df["vac2_2"].str[:1].astype(int)

# For "5 or more" type of response, use top-coding
model_df["household_children"] = pd.to_numeric(
    model_df["household_children"].str.extract(r"(\d+)")[0],
    errors="coerce"
)
model_df["household_size"] = pd.to_numeric(
    model_df["household_size"].str.extract(r"(\d+)")[0],
    errors="coerce"
)

### Part 1.5 Train/Test Split in case for "Test Set Leakage"

In [6]:
from sklearn.model_selection import train_test_split

# Split data once (80% training, 20% testing)
train_df, test_df = train_test_split(
    model_df,
    test_size = 0.2,
    random_state = 5,
    stratify=model_df["target"]
)

# Prepare X and y for later modeling
X_train = train_df[features]
X_test = test_df[features]

y_train = train_df["target"]
y_test = test_df["target"]

# Check results
print("Training set:", train_df.shape)
print("Test set:", test_df.shape)

print("\nTraining target distribution:")
print(train_df["i9_health"].value_counts(normalize=True))

print("\nTest target distribution:")
print(test_df["i9_health"].value_counts(normalize=True))

Training set: (4729, 17)
Test set: (1183, 17)

Training target distribution:
i9_health
Yes    0.78875
No     0.21125
Name: proportion, dtype: float64

Test target distribution:
i9_health
Yes    0.788673
No     0.211327
Name: proportion, dtype: float64


Use ordinal encoding (OrdinalEncoder) to transform ordinal variables:

In [7]:
# Encode ordinal variables as numeric levels
from sklearn.preprocessing import OrdinalEncoder

wellbeing_levels = ["Not at all", "Several days", "More than half the days", "Nearly every day"]
phq4_encoder = OrdinalEncoder(categories=[wellbeing_levels, wellbeing_levels],
                              handle_unknown="use_encoded_value", 
                              unknown_value=np.nan)
X_train[["PHQ4_1", "PHQ4_3"]] = phq4_encoder.fit_transform(X_train[["PHQ4_1", "PHQ4_3"]])
X_test[["PHQ4_1", "PHQ4_3"]] = phq4_encoder.transform(X_test[["PHQ4_1", "PHQ4_3"]])


vac_trust_levels = ["Not at all", "A little", "Moderately", "Very much"]
vac7_encoder = OrdinalEncoder(categories=[vac_trust_levels],
                              handle_unknown="use_encoded_value", 
                              unknown_value=np.nan)
X_train[["vac7"]] = vac7_encoder.fit_transform(X_train[["vac7"]])
X_test[["vac7"]] = vac7_encoder.transform(X_test[["vac7"]])

gov_eval_levels = ["Very badly", "Somewhat badly", "Somewhat well", "Very well"]
wcrex1_encoder = OrdinalEncoder(categories=[gov_eval_levels],
                                handle_unknown="use_encoded_value", 
                                unknown_value=np.nan)
X_train[["WCRex1"]] = wcrex1_encoder.fit_transform(X_train[["WCRex1"]])
X_test[["WCRex1"]] = wcrex1_encoder.fit_transform(X_test[["WCRex1"]])

outdoor_freq_levels = ["Not at all", "Rarely", "Sometimes", "Frequently", "Always"]
vent3_encoder = OrdinalEncoder(categories=[outdoor_freq_levels],
                               handle_unknown="use_encoded_value", 
                               unknown_value=np.nan)
X_train[["Vent_3"]] = vent3_encoder.fit_transform(X_train[["Vent_3"]])
X_test[["Vent_3"]] = vent3_encoder.transform(X_test[["Vent_3"]])

Use one-hot encoding (OneHotEncoder) for categorical variables:

In [8]:
from sklearn.preprocessing import OneHotEncoder
onehot_cols = ["region", "employment_status"]
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

train_encoded = pd.DataFrame(
    enc.fit_transform(X_train[onehot_cols]),
    columns=enc.get_feature_names_out(onehot_cols),
    index=X_train.index
)
test_encoded = pd.DataFrame(
    enc.transform(X_test[onehot_cols]),
    columns=enc.get_feature_names_out(onehot_cols),
    index=X_test.index
)

X_train = pd.concat([X_train.drop(columns=onehot_cols), train_encoded], axis=1)
X_test = pd.concat([X_test.drop(columns=onehot_cols), test_encoded], axis=1)

In [9]:
X_train.head()

,age,cantril_ladder,gender,household_size,household_children,qweek,PHQ4_1,PHQ4_3,vac2_1,vac2_2,...,region_Quebec / Québec,region_Saskatchewan,region_Yukon,employment_status_Full time employment,employment_status_Full time student,employment_status_Not working,employment_status_Other,employment_status_Part time employment,employment_status_Retired,employment_status_Unemployed
3324,52,6,0,2.0,0.0,47,0.0,0.0,3,2,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
5739,34,6,0,2.0,0.0,50,1.0,3.0,2,4,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
5205,51,5,0,1.0,0.0,49,NaN,NaN,3,3,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1894,34,5,0,2.0,1.0,46,0.0,2.0,5,5,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1392,70,8,1,2.0,0.0,45,1.0,1.0,1,5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
